<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L07-an-environment-for-a-tool-using-agent/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L07-an-environment-for-a-tool-using-agent/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T04-L07-an-environment-for-a-tool-using-agent/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T04-L07-an-environment-for-a-tool-using-agent/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T04-L07 · An environment for a tool-using agent: reset, step and reward, and a group-relative update that learns which tool to call

**You will build:** a small environment in the shape that the environment hub this lesson follows (OpenEnv)
uses for reinforcement learning (`reset`, `step`, an observation that carries `done` and `reward`), twenty multi-step
tool tasks to run in it, and a tabular softmax policy trained on those tasks with a group-relative
advantage update. Then you will measure what the environment's *reward* does to the agent: the same
training run, pointed at a reward for outcomes and at a reward for tool calls.

**Time:** ~2 hours · **Runs on:** a laptop CPU, no GPU, no download, no network ·
**Prerequisites:** T04-L01-grpo-from-scratch, T07-L01-agent-loop-from-scratch.

By the end you will be able to:

1. Implement the dynamics, the reward and the `reset` / `step` / `state` interface of a small tool
   environment, following the conventions of the environment hub this lesson follows (OpenEnv), and prove them
   on a scripted episode.
2. Implement episode collection with a softmax policy, the group-relative advantage with its
   zero-variance case, the policy gradient of a tabular softmax, and a group-relative training loop.
3. Measure success rate, average steps and the fraction of groups that teach nothing, over a fixed set
   of seeds, as a policy learns which tool to call.
4. Measure the reward-hacking rate when the environment pays for tool calls instead of outcomes, and
   show that a reward for progress that cannot be repeated does not have the same loophole.
5. Explain what this toy shows and does not show about training real agents in environments.

**This is a toy.** The policy is a table with a few hundred entries and the "tools" are a dictionary.
That is deliberate: the subject here is the environment and its reward, not the model. This notebook
does not import any environment library; it follows the conventions the sources in `claims.yaml` describe.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T04-L01`, `T07-L01`

**This lesson assumes you can already:**

- write Python functions and classes, and index numpy arrays with integer arrays
- say what a probability distribution over actions is, and draw a sample from one with a single uniform random number
- say what an episode, a state, an action and a reward are in reinforcement learning
- compute a mean, and say why a reward is averaged over a batch

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Gymnasium documentation, Basic Usage](https://gymnasium.farama.org/introduction/basic_usage/) · Farama Foundation · *free* — shows reset and step, and the terminated and truncated flags, the convention the environment you write follows
- [OpenEnv documentation](https://huggingface.co/docs/openenv) · the OpenEnv project (github.com/openenv; documentation hosted by Hugging Face) · *free* — the environment hub whose reset, step and observation conventions this lesson mirrors, and where its current interface is documented
- [Spinning Up in Deep RL: Part 3, Intro to Policy Optimization](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html) · OpenAI (Joshua Achiam) · *free* — derives the gradient of the log-probability of a sampled action and why a baseline such as a group mean leaves it unbiased, which is exercises 6 and 7
- [DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models](https://arxiv.org/abs/2402.03300) · Shao et al. (arXiv) · *free* — section 4.1 introduces the group-relative advantage that exercise 6 implements
- [Reward Hacking in Reinforcement Learning](https://lilianweng.github.io/posts/2024-11-28-reward-hacking/) · Lilian Weng (Lil'Log) · *free* — surveys how an agent exploits a flawed reward, the failure that sections 16 and 17 plant on purpose and measure

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import sys
import time
import traceback
from dataclasses import dataclass, field
from typing import Any, Callable, NamedTuple, Optional

import numpy as np

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20261019           # this lesson's one fixed seed: every task, sample and training run derives from it
TASK_SEED = 7             # the seed that draws the task list, once, in make_tasks
OPENENV_REVISION = "v0.9.0"      # the environment-hub release the conventions below were read from
OPENENV_DATE = "2026-10-20"      # the day claims.yaml was checked ("as of" this date)

TOOLS = ["search", "open_page", "read_file", "grep", "calc", "sql", "summarise", "write_note"]
SUBMIT = len(TOOLS)       # action 8 is not a tool: it hands in the answer and ends the episode
N_ACTIONS = len(TOOLS) + 1
N_GOALS = 5               # a task needs some of these sub-goals, in increasing order
DONE_GOAL = N_GOALS       # "pending goal" once every needed sub-goal is met: only submit is left
MAX_STEPS = 12            # an episode that has not submitted by now is truncated
DOMAINS = ["web", "files", "data", "maths"]
# GOAL_TOOL[d][g] is the one tool that meets sub-goal g in domain d (injective within a domain)
GOAL_TOOL = [[0, 1, 6, 4, 7], [3, 2, 6, 4, 7], [5, 6, 3, 4, 7], [4, 0, 3, 6, 7]]
N_STATES = len(DOMAINS) * (N_GOALS + 1)    # what the policy sees: (domain, pending goal)
SOLVED_REWARD = 1.0       # outcome reward for a correct submit
CALL_REWARD = 0.1         # per-call reward in the "calls" and "progress" modes
HACK_REPEATS = 3          # a truncated episode with at least this many repeated calls counts as hacking
MODES = ("outcome", "calls", "progress")
ADV_EPS = 1e-4            # added to the group standard deviation
GROUP = 8                 # completions per task in a group
LR = 30.0
ITERATIONS = 60
SEEDS = (0, 1, 2, 3, 4)   # the five training seeds the measurements use, fixed before any comparison
EVAL_PER_TASK = 40        # evaluation episodes per task

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("transition",),
    "exercise 2": ("step_reward",),
    "exercise 3": ("reset", "step"),
    "exercise 4": ("softmax_sample",),
    "exercise 5": ("run_episode",),
    "exercise 6": ("group_advantages", "zero_variance_fraction"),
    "exercise 7": ("policy_gradient",),
    "exercise 8": ("grpo_update",),
    "exercise 9": ("train_grpo",),
    "exercise 10": ("episode_metrics",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (reset, step)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: it compares the process's resident memory with what it was when the
#    block started, and stops the block as soon as the growth passes `memory_mib`. After a memory
#    stop it looks every 5 ms and stops again at once only if the memory is still growing (a bare
#    `except:` swallowed the stop; clean-up that itself allocates over 32 MiB looks the same);
#    otherwise the stop repeats after 0.25 s, as a time stop does, so `finally:` blocks and
#    context managers can clean up. It looks after 0.01 s,
#    then every 0.02 s, and more often (down to every 2 ms) once the growth passes a quarter of
#    the limit or is fast enough to get there soon: numpy can add a gigabyte in a tenth of a
#    second. A loop that appends to a list reaches gigabytes in seconds, long before
#    any sensible time limit, so the time limit alone does not bound it. The limit is 768 MiB:
#    mybinder.org gives a whole notebook 2 GB, the heaviest lesson holds about 800 MiB before a
#    check starts, and a runaway check runs twice in "Run all" (its cell, then the board). The
#    largest growth a correct answer needs in any lesson is 512 MiB. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and (not state["fired"] or state.get("memory")):
            due.append(now + state.get("poll", 0.01))    # and look at the memory: first after 0.01 s, then as _fire says
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and (not state["fired"] or state.get("memory")) else 0
        if used_mib and not state["fired"]:              # close to the limit, or fast enough to come close soon
            rate = (grown - state.get("grown", 0)) / max(now - state.get("seen", started), 0.001)   # MiB a second
            state["grown"], state["seen"] = grown, now
            state["poll"] = (0.02 if grown <= memory_mib / 4 and rate * 0.02 <= (memory_mib - grown) / 4 else   # near: by
                                         max(0.002, min(0.01, (memory_mib - grown) / max(rate, 1.0) / 4)))  # a 1/4 of the time left
        if grown > memory_mib and state["next"] is not None and not state["fired"]:   # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
            state["memory"], state["at_stop"], state["poll"] = True, grown, 0.005   # then look every 5 ms
        elif state.get("memory") and state["fired"] and grown - state["at_stop"] > 32 and state["next"] is not None:
            state["next"], state["at_stop"] = now, grown        # still growing: a bare `except:` swallowed the stop
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 90     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 768      # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The idea: the environment is the unit, and its reward is part of it

Training a tool-using agent with reinforcement learning needs something that is not the model: an
**environment**. It starts an episode (`reset`), takes the agent's action and says what happened
(`step`), and reports the situation (`state`). The hub this lesson follows, OpenEnv, uses a Gymnasium-style
shape so that one training framework can run many environments. The interface is still changing, so
this lesson pins the revision it read: the cell below prints it, and `claims.yaml` has the sources with
their access date.

Three conventions matter here, and the environment you build follows all three:

- **`reset()` returns an observation, and `step(action)` returns an observation too.** The hub's
  observation carries `done`, `reward` and `metadata` itself, where Gymnasium returns a tuple.
- **The reward is part of the environment.** The agent never chooses it. If it pays for the wrong thing,
  the agent will find out before you do (sections 16 and 17).
- **An episode can end in two ways:** the task ended (here: the agent submitted), or the step cap cut it
  short (**truncated**). Gymnasium keeps the two apart, and so does this environment.

The agent is a table of logits over nine actions: eight tools and `submit`. It learns with the
group-relative advantage from T04-L01: run a group of episodes on the same task, and score each one
against the others. The task is deliberately easy for a model and informative for an environment.

In [ ]:
print(f"conventions read from the OpenEnv release {OPENENV_REVISION}, as of {OPENENV_DATE}:")
print("  reset() / step(action) / state, each reset and step returning an Observation")
print("  Observation fields used here: done, reward, metadata (plus this lesson's own: domain, pending)")
print("  Gymnasium splits the end of an episode into terminated and truncated; so does this lesson")
print(f"actions: {len(TOOLS)} tools {TOOLS} and submit (action {SUBMIT}); step cap {MAX_STEPS}")
print(f"rewards: solved {SOLVED_REWARD}, per call {CALL_REWARD}; a truncated episode with at least "
      f"{HACK_REPEATS} repeated calls is counted as hacking")

## 2. The world: tasks that need several tools in order

A task lives in one of the **domains** and needs a few of the **sub-goals**, always in increasing
order (find, read, pick out, work out, save). Each sub-goal is met by exactly one tool *in that domain*.
So the agent has to learn which tool serves which sub-goal where, and when to stop: `submit` before
every needed sub-goal is met ends the episode as a failure.

What the agent is shown after each step is only the **domain** and the **pending sub-goal** (the next
one the task still needs, or "done"). That pair is the policy's state: a table with one row per pair.
The tasks are drawn once, from a fixed seed, by the code in the next cell.

In [ ]:
GOAL_NAMES = ["find", "read", "pick out", "work out", "save"]


def make_tasks(seed: int = TASK_SEED) -> list:
    """Draw the task list: for every domain, five tasks, each a (domain, chain) pair where the
    chain is a sorted tuple of two to four distinct sub-goals."""
    rng = np.random.default_rng(seed)
    tasks = []
    for domain in range(len(DOMAINS)):
        for _ in range(5):
            length = int(rng.integers(2, 5))
            chain = sorted(rng.choice(N_GOALS, size=length, replace=False).tolist())
            tasks.append((domain, tuple(int(g) for g in chain)))
    return tasks


TASKS = make_tasks()
N_TASKS = len(TASKS)


def pending_goal(task: tuple, progress: int) -> int:
    """The next sub-goal the task still needs, or DONE_GOAL when `progress` has met them all."""
    chain = task[1]
    return chain[progress] if progress < len(chain) else DONE_GOAL


def _show_tasks() -> None:
    print(f"{N_TASKS} tasks; each row: task, domain, sub-goals needed, the tools that meet them in order")
    for i, (domain, chain) in enumerate(TASKS):
        tools = " > ".join(TOOLS[GOAL_TOOL[domain][g]] for g in chain)
        print(f"  task {i:>2}  {DOMAINS[domain]:<5}  goals {list(chain)!s:<13}  {tools}")
    lengths = [len(chain) for _, chain in TASKS]
    print(f"calls needed per task: from {min(lengths)} to {max(lengths)} tools, then submit; "
          f"the shortest possible episode averages {np.mean(lengths) + 1:.2f} steps")
    states = len(DOMAINS) * (N_GOALS + 1)
    assert states == N_STATES
    print(f"the policy's state is (domain, pending sub-goal): {N_STATES} rows of {N_ACTIONS} logits "
          f"= {N_STATES * N_ACTIONS} numbers in all")


_show_tasks()

## 3. Exercise 1: the dynamics

Write `transition(task, progress, action)`. `progress` is how many of the task's needed sub-goals are met
so far. It returns the new progress and an **event**, one of five strings:

- `"advanced"`: `action` is a tool and it is the tool that meets the **pending** sub-goal. Progress goes up by one.
- `"repeat"`: `action` is a tool that already met one of the sub-goals **before** the pending one (the
  tool is cached: calling it again works and changes nothing).
- `"error"`: any other tool: it does not meet the pending sub-goal and has not met an earlier one. Progress is unchanged.
- `"solved"`: `action` is `SUBMIT` and `progress` equals the number of needed sub-goals.
- `"gave_up"`: `action` is `SUBMIT` with a sub-goal still pending. Progress is unchanged.

Careful with the neighbours: a tool that meets a *later* sub-goal is an error, not progress; and once
every sub-goal is met, a tool from the chain is a repeat and any other tool is an error.

<details><summary>Hint 1: what to think about</summary>

Say the cases out loud in order: submit first (it is not a tool), then the tool for the pending
sub-goal, then tools for sub-goals already met. Everything else is an error. Which tool serves
which sub-goal is `GOAL_TOOL[domain][goal]`.
</details>

<details><summary>Hint 2: the approach in words</summary>

Handle `SUBMIT` before anything else. Otherwise compare `action` with the tool for the pending
sub-goal (only if one is pending), then with the tools of the sub-goals the task had before that one.
Return the unchanged progress in every case but the first.
</details>

In [ ]:
EVENTS = ("advanced", "repeat", "error", "solved", "gave_up")


def transition(task: tuple, progress: int, action: int) -> tuple:
    """Apply one action to a task in progress.

    Args:
        task: (domain, chain): the domain index and the tuple of sub-goals the task needs, in order.
        progress: how many of those sub-goals are already met.
        action: 0..7 a tool, SUBMIT (8) to hand in.

    Example (domain 0 is "web": goal 1 is met by tool 1, goal 4 by tool 7):
        >>> task = (0, (1, 4))
        >>> transition(task, 0, 1)
        (1, 'advanced')
        >>> transition(task, 1, 1)
        (1, 'repeat')
        >>> transition(task, 0, 7)
        (0, 'error')
        >>> transition(task, 2, SUBMIT)
        (2, 'solved')

    Returns:
        (new_progress, event) with event one of EVENTS.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement transition")


def _check_transition() -> None:
    task = (0, (1, 4))      # web: open_page (1), then write_note (7)
    assert transition(task, 0, 1) == (1, "advanced"), "the tool for the pending sub-goal should advance the task"
    assert transition(task, 1, 7) == (2, "advanced"), "progress should go up by one for each correct tool"
    assert transition(task, 0, 7) == (0, "error"), \
        "a tool that serves a LATER sub-goal is an error: it must not advance the task"
    assert transition(task, 0, 3) == (0, "error"), "a tool that serves no sub-goal here should be an error"
    assert transition(task, 1, 1) == (1, "repeat"), "a tool that already met an earlier sub-goal is a 'repeat'"
    assert transition(task, 2, 7) == (2, "repeat"), "after the last sub-goal, a tool from the chain is a 'repeat'"
    assert transition(task, 2, 1) == (2, "repeat"), \
        "a tool that met ANY sub-goal before the pending one is a 'repeat', not only the most recent one"
    assert transition(task, 2, 3) == (2, "error"), "after the last sub-goal, any other tool is still an error"
    assert transition(task, 2, SUBMIT) == (2, "solved"), "submitting with every sub-goal met is 'solved'"
    assert transition(task, 1, SUBMIT) == (1, "gave_up"), "submitting with a sub-goal pending is 'gave_up'"
    assert transition(task, 0, SUBMIT) == (0, "gave_up"), "submitting at the start is 'gave_up'"
    other = (1, (0, 1))     # files: grep (3), then read_file (2)
    assert transition(other, 0, 3) == (1, "advanced"), "the tool for a sub-goal depends on the domain: GOAL_TOOL[domain][goal]"
    assert transition(other, 0, 0) == (0, "error"), "tool 0 serves sub-goal 0 only in the web domain"
    assert transition(other, 1, 3) == (1, "repeat"), "a 'repeat' is judged with the task's own domain too, not with web's tools"
    assert transition(other, 2, 3) == (2, "repeat"), \
        "after the last sub-goal, a tool from the chain is a 'repeat' in every domain, judged with that domain's tools"
    longer = (2, (0, 1, 2, 4))     # data: these four sub-goals use tools 5, 6, 3 and 7
    assert transition(longer, 1, 3) == (1, "error"), \
        "a tool that serves a LATER sub-goal is an error at any progress, not only at the start"
    assert transition(longer, 1, 7) == (1, "error"), "a tool for a sub-goal further ahead is an error too, at any progress"
    progress, event = transition(task, 0, 1)
    assert type(progress) is int and type(event) is str, "return a (progress, event) pair of a plain int and a string"
    print("exercise 1 looks right")


_try("exercise 1", _check_transition)

## 4. Exercise 2: the reward

Write `step_reward(event, mode)`. The reward is a function of what just happened, and the **mode** says
what the environment's author chose to pay for. Three modes:

- `"outcome"`: `SOLVED_REWARD` for `"solved"`, otherwise `0.0`. Nothing is paid along the way.
- `"calls"`: `CALL_REWARD` for every event in which a tool call worked, that is `"advanced"` and `"repeat"`,
  otherwise `0.0`. Nothing is paid for the outcome. This is the environment of someone who wanted agents
  that use their tools.
- `"progress"`: `CALL_REWARD` for `"advanced"` only, plus `SOLVED_REWARD` for `"solved"`, otherwise `0.0`.

Any other mode raises `ValueError`, whatever the event (a typo in a mode name should stop the run, not train on
zeros). Names are matched exactly: `"Outcome"` and `"calls "` are other modes.
Every return value is a plain Python float, `0.0` and not `0` where nothing is paid (not a numpy float).
Compare the second and third modes now: they differ in one word, and section 16 measures what it costs.

<details><summary>Hint 1: what to think about</summary>

Three branches on `mode`, each a rule about which events earn something. For each mode, which events
earn the call reward, and does the solved reward appear at all?
</details>

<details><summary>Hint 2: the approach in words</summary>

Check the mode first and raise for a name you do not know, even when the event would pay nothing.
Then return the constant for the events the mode pays for and `0.0` for the rest. Return plain Python
floats, `0.0` included, never the integer `0` or a numpy float.
</details>

In [ ]:
def step_reward(event: str, mode: str) -> float:
    """The reward for one event under one reward mode.

    Example:
        >>> step_reward("solved", "outcome"), step_reward("repeat", "outcome")
        (1.0, 0.0)
        >>> step_reward("repeat", "calls"), step_reward("solved", "calls")
        (0.1, 0.0)
        >>> step_reward("advanced", "progress"), step_reward("repeat", "progress")
        (0.1, 0.0)

    Returns:
        A plain Python float (0.0, not 0, where nothing is paid). Raises ValueError for a mode not in MODES.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement step_reward")


def _check_step_reward() -> None:
    for event in EVENTS:
        assert step_reward(event, "outcome") == (SOLVED_REWARD if event == "solved" else 0.0), \
            f"outcome mode pays only for 'solved'; check the event {event!r}"
    assert step_reward("advanced", "calls") == CALL_REWARD, "calls mode pays for a call that advanced the task"
    assert step_reward("repeat", "calls") == CALL_REWARD, "calls mode pays for a repeated call too: that is its whole point"
    assert step_reward("error", "calls") == 0.0 and step_reward("gave_up", "calls") == 0.0, "an error pays nothing"
    assert step_reward("solved", "calls") == 0.0, "calls mode pays nothing for the outcome"
    assert step_reward("advanced", "progress") == CALL_REWARD, "progress mode pays for an advance"
    assert step_reward("repeat", "progress") == 0.0, "progress mode must not pay for a repeat, or it has the same loophole"
    assert step_reward("solved", "progress") == SOLVED_REWARD, "progress mode pays the solved reward as well"
    assert step_reward("error", "progress") == 0.0 and step_reward("gave_up", "progress") == 0.0, \
        "progress mode pays nothing for an error or a give-up"
    for mode in ("outcomes", "Outcome", "calls ", ""):
        for event in EVENTS:
            try:
                step_reward(event, mode)
            except ValueError:
                continue
            raise AssertionError(f"step_reward({event!r}, {mode!r}) should raise ValueError: check the mode before the "
                                 "event, and match its name exactly (no lower-casing, no stripping), not fall through to 0.0")
    for mode in MODES:
        for event in EVENTS:
            assert type(step_reward(event, mode)) is float, (
                f"step_reward({event!r}, {mode!r}) returned {type(step_reward(event, mode)).__name__}: return a "
                "plain Python float (wrap it in float(), and write 0.0, not 0), not a numpy float or an int")
    print("exercise 2 looks right")


_try("exercise 2", _check_step_reward)

## 5. Exercise 3: reset, step and state

Now put the dynamics and the reward behind the interface. The class below is complete except for two
methods. The skeleton already stores what you need: `self._task_id` (index into `self.tasks`),
`self._progress`, `self._steps` and `self._finished`, and `self._observe(reward, done, metadata)` builds
the `Observation` from them. `state` is done for you.

- **`reset(seed=None, episode_id=None, task_id=0)`** starts an episode on task `task_id`: progress and steps back
  to zero, not finished, an episode id (`episode_id` if given, otherwise `"episode-N"`, where N counts every reset
  on this object from 1, including one given an explicit `episode_id`). It returns `self._observe(None, False, {})`:
  the reward of a first observation is `None`, not `0.0`. `seed` is accepted for the interface and ignored: this world is deterministic.
- **`step(action)`** applies `transition`, then `step_reward` with `self.reward_mode`, and counts the step.
  The episode **ends** when the event is `"solved"` or `"gave_up"`, and is **truncated** when it has not ended
  and the step count has reached `MAX_STEPS`; `done` is true in both cases. It returns the observation with the
  float reward, `done`, and `metadata={"event": event, "truncated": truncated}`, where `done` and `truncated` are
  plain Python bools (`True` or `False`, not `1` or `0`). Calling `step` before `reset`, or after `done`, raises
  `RuntimeError`.

<details><summary>Hint 1: what to think about</summary>

`reset` is bookkeeping: set every field the skeleton lists. In `step`, the order matters: the step count
must include the step just taken before you compare it with the cap, and a submit on the last allowed
step is a finished episode, not a truncated one.
</details>

<details><summary>Hint 2: the approach in words</summary>

In `step`: refuse if there is no live episode; call `transition` with the current task and progress and store the
new progress; add one to the step count; ask `step_reward` for the reward; set `ended` from the event and
`truncated` from `ended` and the step count; remember whether the episode is now finished; return `_observe`.
</details>

In [ ]:
@dataclass
class Observation:
    """What `reset` and `step` return. `done`, `reward` and `metadata` are the fields of the hub's base
    observation; `domain`, `pending` and `state_id` are this lesson's, and `message` is for people."""
    done: bool = False
    reward: Optional[float] = None
    metadata: dict = field(default_factory=dict)
    domain: int = 0
    pending: int = DONE_GOAL
    state_id: int = 0
    message: str = ""


@dataclass
class State:
    """What `state` reports about the episode in progress."""
    episode_id: str = ""
    step_count: int = 0
    task_id: int = -1
    progress: int = 0


class ToolEnv:
    """A tool environment with the hub's interface: reset(), step(action), state."""

    def __init__(self, reward_mode: str = "outcome", tasks: Optional[list] = None):
        if reward_mode not in MODES:
            raise ValueError(f"unknown reward mode {reward_mode!r}")
        self.reward_mode = reward_mode
        self.tasks = TASKS if tasks is None else tasks
        self._task_id: Optional[int] = None     # which task is running; None before the first reset
        self._progress = 0                      # sub-goals met so far
        self._steps = 0                         # actions taken so far
        self._finished = True                   # True when no episode is live (before reset, and after done)
        self._episode = ""                      # the episode id
        self._resets = 0                        # how many times reset() has been called

    def _observe(self, reward, done: bool, metadata: dict) -> Observation:
        task = self.tasks[self._task_id]
        pending = pending_goal(task, self._progress)
        need = "nothing: submit" if pending == DONE_GOAL else GOAL_NAMES[pending]
        event = metadata.get("event")
        message = f"{DOMAINS[task[0]]} task; still needed: {need}" + (f"; last call: {event}" if event else "")
        return Observation(done, reward, metadata, task[0], pending, task[0] * (N_GOALS + 1) + pending, message)

    def reset(self, seed: Optional[int] = None, episode_id: Optional[str] = None, task_id: int = 0) -> Observation:
        """Start an episode on task `task_id` and return the first observation.

        Example:
            >>> env = ToolEnv(); obs = env.reset(task_id=0)
            >>> obs.done, obs.reward, env.state.step_count, env.state.episode_id
            (False, None, 0, 'episode-1')

        Returns:
            An Observation with reward None, done False and empty metadata.
        """
        # YOUR CODE HERE
        raise NotImplementedError("implement reset")

    def step(self, action: int) -> Observation:
        """Take one action and return the observation, with its reward and `done`.

        Example (task 1 is a web task whose first needed tool is tool 1):
            >>> env = ToolEnv(); _ = env.reset(task_id=1); obs = env.step(5)
            >>> obs.metadata, obs.reward, obs.done
            ({'event': 'error', 'truncated': False}, 0.0, False)

        Returns:
            An Observation: float reward, done, metadata {"event": ..., "truncated": ...}.
            Raises RuntimeError before reset() and after done.
        """
        # YOUR CODE HERE
        raise NotImplementedError("implement step")

    @property
    def state(self) -> State:
        return State(self._episode, self._steps, -1 if self._task_id is None else self._task_id, self._progress)


def oracle_action(obs: Observation) -> int:
    """The action a perfect agent takes: the tool for the pending sub-goal, or submit when none is pending."""
    return SUBMIT if obs.pending == DONE_GOAL else GOAL_TOOL[obs.domain][obs.pending]


def _check_env() -> None:
    env = ToolEnv("outcome")
    try:
        env.step(0)
    except RuntimeError:
        pass
    else:
        raise AssertionError("step() before reset() should raise RuntimeError")
    obs = env.reset(task_id=2)
    assert obs.reward is None and obs.done is False and obs.metadata == {}, \
        "reset returns an observation with reward None (not 0.0), done False and empty metadata"
    assert env.state.step_count == 0 and env.state.task_id == 2 and env.state.progress == 0, "reset should clear the counters"
    assert env.state.episode_id == "episode-1", "the default episode id is 'episode-N', N counting every reset on this object from 1"
    assert obs.domain == TASKS[2][0] and obs.pending == TASKS[2][1][0] and obs.state_id == TASKS[2][0] * (N_GOALS + 1) + TASKS[2][1][0], \
        "reset's observation shows the task you asked for: its domain and its first needed sub-goal (build it with _observe)"
    steps, events = 0, []
    while not obs.done:
        obs = env.step(oracle_action(obs))
        steps += 1
        events.append(obs.metadata["event"])
    assert events == ["advanced"] * len(TASKS[2][1]) + ["solved"], \
        f"a perfect agent's events should be one 'advanced' per needed sub-goal, then 'solved'; got {events}"
    assert obs.metadata == {"event": "solved", "truncated": False}, "a perfect agent should end with event 'solved'"
    assert obs.reward == SOLVED_REWARD and env.state.step_count == steps == len(TASKS[2][1]) + 1, \
        "a perfect agent needs one step per needed sub-goal plus the submit; the outcome reward arrives on the last one"
    try:
        env.step(0)
    except RuntimeError:
        pass
    else:
        raise AssertionError("step() after done should raise RuntimeError")
    obs = env.reset(task_id=0, episode_id="abc")
    assert env.state.episode_id == "abc" and env.state.step_count == 0, "reset should accept an episode id and restart the count"
    for k in range(MAX_STEPS):
        assert not obs.done, f"the episode ended after {k} steps; the cap is MAX_STEPS = {MAX_STEPS}"
        assert k == 0 or obs.metadata["truncated"] is False, f"after {k} of {MAX_STEPS} steps the episode is live, not truncated"
        obs = env.step(5)
        if k == 0:
            assert obs.reward == 0.0 and obs.metadata == {"event": "error", "truncated": False} and env.state.progress == 0, \
                "a failed call: reward 0.0, metadata {'event': 'error', 'truncated': False}, progress unchanged"
    assert obs.done and obs.metadata["truncated"] and obs.metadata["event"] == "error", \
        "an episode that reaches MAX_STEPS without submitting is done and truncated"
    assert env.state.step_count == MAX_STEPS, "the truncating step is counted: step_count equals MAX_STEPS"
    assert type(obs.done) is bool and type(obs.metadata["truncated"]) is bool, (
        f"done has type {type(obs.done).__name__} and truncated type {type(obs.metadata['truncated']).__name__}: both are "
        "plain Python bools (True or False, not 1 or 0, not a numpy bool)")
    try:
        env.step(0)
    except RuntimeError:
        pass
    else:
        raise AssertionError("step() after a truncated episode should raise RuntimeError: done covers every way an "
                             "episode ends, so mark it finished when it is truncated as well")
    obs = env.reset(task_id=0)
    assert env.state.episode_id == "episode-3", \
        "N in 'episode-N' counts every reset on this object, including one given an explicit episode_id"
    for _ in range(MAX_STEPS - 1):
        obs = env.step(5)
    obs = env.step(SUBMIT)
    assert obs.done and obs.metadata == {"event": "gave_up", "truncated": False}, \
        "a submit ends the episode; it is 'truncated' only when the cap cut the episode short"
    try:
        env.step(0)
    except RuntimeError:
        pass
    else:
        raise AssertionError("step() after a given-up episode should raise RuntimeError: done covers every way an "
                             "episode ends, so mark it finished when it gives up as well")
    obs = env.reset(task_id=0)
    for _ in range(MAX_STEPS - len(TASKS[0][1]) - 1):
        obs = env.step(5)
    while obs.pending != DONE_GOAL:
        obs = env.step(oracle_action(obs))
    obs = env.step(SUBMIT)
    assert env.state.step_count == MAX_STEPS and obs.metadata == {"event": "solved", "truncated": False}, \
        "a submit that solves the task on the last allowed step ends the episode: it is solved, not truncated"
    calls = ToolEnv("calls")
    first = calls.reset(task_id=0)
    live = calls.step(oracle_action(first))
    assert live.reward == CALL_REWARD, "step should use the env's reward_mode"
    assert type(live.reward) is float, "the reward in a step's observation is a plain Python float"
    assert live.pending == TASKS[0][1][1] and calls.state.progress == 1 and calls.state.step_count == 1, \
        "an advance moves the pending sub-goal on by one and counts the step"
    assert type(live.done) is bool and type(live.metadata["truncated"]) is bool, (
        f"done has type {type(live.done).__name__} and truncated type {type(live.metadata['truncated']).__name__} on a live "
        "step: both are plain Python bools (False here, not 0, not a numpy bool)")
    first = calls.reset(task_id=0)
    assert calls.state.step_count == 0 and calls.state.progress == 0 and first.pending == TASKS[0][1][0], \
        "a reset in the middle of an episode starts again from scratch: step count and progress back to zero"
    for _ in range(MAX_STEPS - 1):
        calls.step(5)
    calls.reset(task_id=0)
    assert calls.step(5).done is False, "after a reset the step count restarts, so the cap is not reached early"
    progress_env, total = ToolEnv("progress"), 0.0
    obs = progress_env.reset(task_id=0)
    first_tool = oracle_action(obs)
    while not obs.done:
        obs = progress_env.step(oracle_action(obs))
        total += obs.reward
    assert abs(total - (len(TASKS[0][1]) * CALL_REWARD + SOLVED_REWARD)) < 1e-9, \
        "in progress mode a perfect agent is paid for every advance and for the solve: step pays step_reward(event, reward_mode)"
    progress_env.reset(task_id=0)
    paid = progress_env.step(first_tool).reward + progress_env.step(first_tool).reward
    assert abs(paid - CALL_REWARD) < 1e-12, "in progress mode an advance pays and the repeat after it does not"
    print("exercise 3 looks right")


_try("exercise 3", _check_env, needs=("exercise 1", "exercise 2"))

In [ ]:
def _show_scripted() -> None:
    env = ToolEnv("outcome")
    obs = env.reset(task_id=0)
    print(f"task 0: {obs.message}")
    while not obs.done:
        action = oracle_action(obs)
        obs = env.step(action)
        name = "submit" if action == SUBMIT else TOOLS[action]
        print(f"  step {env.state.step_count}: {name:<10} -> {obs.metadata['event']:<8} reward {obs.reward}  | {obs.message}")
    print(f"the perfect agent finishes in {env.state.step_count} steps, truncated: {obs.metadata['truncated']}")


_try("scripted episode", _show_scripted, needs=("exercise 1", "exercise 2", "exercise 3"))

## 6. Exercise 4: choosing an action from the policy

The policy is a table `table[state_id]` of nine logits. Write `softmax_sample(logits, u)`: turn the
logits into probabilities with a **numerically stable** softmax (subtract the largest logit first) and
return the **smallest index `k` whose cumulative probability is at least `u`**, where `u` is a uniform
draw in [0, 1) that the caller supplies. If rounding leaves no such index (the cumulative sum ends a hair
below `u`), return the last index. Taking `u` as an argument keeps the function exact: the same `u` always
gives the same action, which is what makes a training run repeatable.

<details><summary>Hint 1: what to think about</summary>

Sampling from a discrete distribution with one uniform number: lay the probabilities end to end on
[0, 1] and see which one `u` lands in. Logits of a thousand overflow `exp` unless you shift them first.
</details>

<details><summary>Hint 2: the approach in words</summary>

Shift by the maximum, exponentiate, divide by the sum, take the cumulative sum. Count how many cumulative
values are strictly below `u`: that count is the index you want, capped at the last index.
</details>

In [ ]:
def softmax_sample(logits: np.ndarray, u: float) -> int:
    """Return the action whose slice of the cumulative probability contains `u`.

    Example:
        >>> softmax_sample(np.array([0.0, 0.0]), 0.25), softmax_sample(np.array([0.0, 0.0]), 0.75)
        (0, 1)
        >>> softmax_sample(np.array([0.0, 0.0]), 0.5)      # the first index whose cumulative sum reaches u
        0

    Returns:
        An int in range(len(logits)).
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement softmax_sample")


def _check_softmax_sample() -> None:
    flat = np.zeros(4)
    assert [softmax_sample(flat, u) for u in (0.0, 0.24, 0.26, 0.74, 0.76, 0.99)] == [0, 0, 1, 2, 3, 3], \
        "equal logits should split [0, 1) into equal slices, in index order"
    assert softmax_sample(np.zeros(2), 0.5) == 0, "when u equals a cumulative sum exactly, take the smaller index"
    assert softmax_sample(np.array([0.0, 1000.0]), 0.5) == 1, \
        "a logit of 1000 overflows exp: subtract the largest logit first"
    assert softmax_sample(np.array([1000.0, 0.0]), 0.999999) == 0, "all of the mass is on index 0"
    assert softmax_sample(np.array([0.0, 0.0, 0.0]), 1.0) == 2, "u at the top of the range should give the last index, not run past it"
    assert softmax_sample(np.array([1000.0, 1000.0, 0.0]), 0.75) == 1, \
        "two logits of 1000 overflow exp to inf, and inf / inf is nan: subtract the largest logit first"
    assert softmax_sample(np.zeros(10), 1.0) == 9, \
        "ten equal slices add up to a hair below 1.0, so no cumulative sum reaches u = 1.0: cap the index at the last one"
    hits = np.bincount([softmax_sample(np.log([0.1, 0.6, 0.3]), u) for u in (np.arange(1000) + 0.5) / 1000], minlength=3)
    assert list(hits) == [100, 600, 300], "the slice of each index should be as wide as its probability"
    assert type(softmax_sample(flat, 0.3)) is int, "return a plain int (not a numpy integer, not a bool)"
    print("exercise 4 looks right")


_try("exercise 4", _check_softmax_sample)

## 7. Exercise 5: running one episode

An **episode** is what the trainer learns from: which state the agent was in, which action it took, what
happened. Write `run_episode(env, table, task_id, rng)`: reset `env` on `task_id`, then until the observation
is `done` read the policy's row for `obs.state_id`, draw **exactly one** `rng.random()` per step, pick the
action with `softmax_sample`, and step. Return an `Episode`:

- `states`, `actions`, `events`: one entry per step (the event string comes from `obs.metadata["event"]`);
- `reward`: the sum of the step rewards (`obs.reward`); `steps`: the number of steps;
- `solved`: whether the **last** event is `"solved"`; `truncated`: the last observation's `metadata["truncated"]`.
  Both are plain Python bools.

The state recorded for a step is the one the agent saw **before** acting.

<details><summary>Hint 1: what to think about</summary>

Two things go wrong silently: recording the state after the step instead of before it, and drawing a
random number more or less than once per step (which shifts every later episode that shares the generator).
</details>

<details><summary>Hint 2: the approach in words</summary>

Keep four lists. Loop while the current observation is not done: remember `obs.state_id`, sample, step, then
append the action, the event, and add the reward. Build the `Episode` from the lists and the final observation.
</details>

In [ ]:
class Episode(NamedTuple):
    states: list
    actions: list
    events: list
    reward: float
    steps: int
    solved: bool
    truncated: bool


def run_episode(env: ToolEnv, table: np.ndarray, task_id: int, rng: np.random.Generator) -> Episode:
    """Play one episode of task `task_id` with the softmax policy `table` and return what happened.

    Example (a table that is certain to pick tool 0 in every state, on a web task that needs tool 0 at no point):
        >>> t = np.full((N_STATES, N_ACTIONS), -50.0); t[:, 0] = 50.0
        >>> ep = run_episode(ToolEnv(), t, 1, np.random.default_rng(0))
        >>> ep.steps, ep.truncated, ep.solved, set(ep.events)
        (12, True, False, {'error'})

    Returns:
        An Episode (see above).
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement run_episode")


def oracle_table() -> np.ndarray:
    """A table that picks the perfect action in every state, with near certainty."""
    table = np.zeros((N_STATES, N_ACTIONS))
    for d in range(len(DOMAINS)):
        for g in range(N_GOALS + 1):
            table[d * (N_GOALS + 1) + g, SUBMIT if g == DONE_GOAL else GOAL_TOOL[d][g]] = 60.0
    return table


def _check_run_episode() -> None:
    env = ToolEnv("outcome")
    rng = np.random.default_rng(3)
    ep = run_episode(env, oracle_table(), 0, rng)
    chain = TASKS[0][1]
    assert ep.steps == len(chain) + 1 and ep.solved and not ep.truncated, \
        "a policy that always takes the perfect action should solve task 0 in one step per sub-goal plus the submit"
    assert ep.events == ["advanced"] * len(chain) + ["solved"], "record obs.metadata['event'] after each step"
    assert ep.reward == SOLVED_REWARD, "reward is the sum of the step rewards"
    assert len(ep.states) == len(ep.actions) == len(ep.events) == ep.steps, "one state, action and event per step"
    assert ep.states[0] == TASKS[0][0] * (N_GOALS + 1) + chain[0], \
        "the state of a step is the one the agent saw BEFORE acting (the observation's state_id)"
    assert ep.actions[-1] == SUBMIT and ep.actions[0] == GOAL_TOOL[TASKS[0][0]][chain[0]], "record the actions taken"
    spare = np.random.default_rng(3)
    for _ in range(ep.steps):
        spare.random()
    assert rng.random() == spare.random(), "draw exactly one rng.random() per step: any more or fewer shifts later episodes"
    lost = np.full((N_STATES, N_ACTIONS), -50.0)
    lost[:, 5] = 50.0
    ep = run_episode(env, lost, 0, np.random.default_rng(0))
    assert ep.steps == MAX_STEPS and ep.truncated and not ep.solved and ep.reward == 0.0, \
        "an agent that never submits runs to the step cap: truncated, not solved"
    ep = run_episode(env, oracle_table(), 0, np.random.default_rng(1))
    again = run_episode(env, oracle_table(), 0, np.random.default_rng(1))
    assert ep == again, "the same table, task and generator state must give the same episode"
    paid = run_episode(ToolEnv("calls"), oracle_table(), 0, np.random.default_rng(3))
    assert abs(paid.reward - len(chain) * CALL_REWARD) < 1e-9, \
        "in calls mode a perfect agent is paid on every step that advanced: the episode reward adds up every step's reward"
    assert paid.solved, "solved means the last event is 'solved', whatever the reward added up to (here it is below 0.5)"
    k = next(t for t in range(N_TASKS) if TASKS[t][0] != TASKS[0][0] and len(TASKS[t][1]) != len(chain))
    table = oracle_table()
    kept = table.copy()
    other = run_episode(env, table, k, np.random.default_rng(3))
    assert other.steps == len(TASKS[k][1]) + 1 and other.solved and other.states[0] == TASKS[k][0] * (N_GOALS + 1) + TASKS[k][1][0], \
        f"reset the environment on the task_id you were given (task {k} here), not on a default task"
    assert np.array_equal(table, kept), "run_episode only reads the policy table: do not change it"
    quitter = np.full((N_STATES, N_ACTIONS), -50.0)
    quitter[:, SUBMIT] = 50.0
    gave = run_episode(env, quitter, 0, np.random.default_rng(3))
    assert gave.events == ["gave_up"] and not gave.solved and not gave.truncated, \
        "an agent that submits early gave up: the episode ended, so it is neither solved nor truncated"

    class _Stalled(ToolEnv):
        """Task 0 with the first `stall` tool calls failing (tool 5 serves none of its sub-goals); then the agent's own."""

        def __init__(self, stall):
            super().__init__("outcome")
            self.stall, self.calls = stall, 0

        def reset(self, *args, **kwargs):
            self.calls = 0
            return super().reset(*args, **kwargs)

        def step(self, action):
            self.calls += 1
            return super().step(5 if self.calls <= self.stall else action)

    late = run_episode(_Stalled(MAX_STEPS - 1 - len(chain)), oracle_table(), 0, np.random.default_rng(3))
    assert late.steps == MAX_STEPS and late.solved and not late.truncated, \
        ("a submit that solves the task on step MAX_STEPS ends the episode, it is not cut off: truncated is the last "
         "observation's metadata['truncated'], not a count of the steps")
    quit_late = run_episode(_Stalled(MAX_STEPS - 1), quitter, 0, np.random.default_rng(3))
    assert quit_late.steps == MAX_STEPS and quit_late.events[-1] == "gave_up", "the stalled agent submits on step MAX_STEPS"
    assert not quit_late.solved and not quit_late.truncated, \
        ("a give-up on step MAX_STEPS ended the episode, it was not cut off: truncated is the last observation's "
         "metadata['truncated'], not 'did not solve and used every step'")
    assert all(type(v) is bool for v in (late.solved, late.truncated, quit_late.solved, quit_late.truncated)), \
        "solved and truncated are plain Python bools (wrap them in bool()), not numpy bools or ints"
    noisy = np.random.default_rng(7).normal(scale=1.5, size=(N_STATES, N_ACTIONS))
    ep = run_episode(ToolEnv("outcome"), noisy, 4, np.random.default_rng(5))
    draws, replay = np.random.default_rng(5), ToolEnv("outcome")
    obs = replay.reset(task_id=4)
    for s, a, e in zip(ep.states, ep.actions, ep.events):
        assert s == obs.state_id, "each recorded state is the observation the agent saw BEFORE its action"
        assert a == softmax_sample(noisy[s], draws.random()), \
            "each action is softmax_sample(table[state], u) with u that step's own rng.random(), drawn in order"
        obs = replay.step(a)
        assert e == obs.metadata["event"], "each event is obs.metadata['event'] of the step just taken"
    assert obs.done and ep.steps == len(ep.actions), "play until the observation is done"
    loop = np.zeros((N_STATES, N_ACTIONS))
    loop[:, GOAL_TOOL[TASKS[0][0]][chain[0]]] = 60.0
    looped = run_episode(ToolEnv("calls"), loop, 0, np.random.default_rng(3))
    assert looped.truncated and not looped.solved and abs(looped.reward - MAX_STEPS * CALL_REWARD) < 1e-9, \
        ("an agent that repeats one working tool is paid on every step in calls mode, and still never solves: solved comes "
         "from the last event and reward adds up every step")
    print("exercise 5 looks right")


_try("exercise 5", _check_run_episode, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 8. Exercise 6: the group-relative advantage

The group-relative advantage from T04-L01, in the setting of this lesson. The batch holds `G` episodes
for each task, **group after group** (all of task 0's, then task 1's, ...). For each group, the advantage
of an episode is its reward minus the group mean, divided by the group's **population** standard
deviation (`ddof=0`) plus `ADV_EPS` (added to the standard deviation, not to the variance).

Write `group_advantages(rewards, group_size)` returning a flat array in the same order, and
`zero_variance_fraction(rewards, group_size)`: the fraction, as a plain Python `float` (not a numpy float), of
groups in which **every reward is exactly equal to the group's first**. Such a group has advantage zero everywhere: it teaches the update nothing.
Neither function changes `rewards` (it may be a numpy array the caller still needs).
The equality is exact on purpose: rewards here are sums of a few constants.

<details><summary>Hint 1: what to think about</summary>

A group is a block of `group_size` consecutive rewards. What happens to the advantage when every
reward in the block is the same, and why does the epsilon matter there?
</details>

<details><summary>Hint 2: the approach in words</summary>

Reshape to (groups, group_size), take the mean and the standard deviation along the second axis, keep
dimensions so they broadcast back, and flatten the result. For the fraction, compare every column with
the first column and ask which rows agree everywhere.
</details>

In [ ]:
def group_advantages(rewards, group_size: int) -> np.ndarray:
    """Within each group of `group_size` consecutive rewards: (reward - mean) / (std + ADV_EPS).

    Example:
        >>> np.round(group_advantages([1.0, 0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5], 4), 2)
        array([ 1.73, -0.58, -0.58, -0.58,  0.  ,  0.  ,  0.  ,  0.  ])

    Returns:
        A float array with one advantage per reward, in the input order.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement group_advantages")


def zero_variance_fraction(rewards, group_size: int) -> float:
    """The fraction of groups whose rewards are all exactly equal.

    Example:
        >>> zero_variance_fraction([1.0, 1.0, 0.0, 1.0], 2)
        0.5

    Returns:
        A plain Python float in [0, 1] (not a numpy float).
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement zero_variance_fraction")


def _check_advantages() -> None:
    adv = group_advantages([1.0, 0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.5], 4)
    std = np.sqrt(0.1875)
    assert np.allclose(adv[:4], np.array([0.75, -0.25, -0.25, -0.25]) / (std + ADV_EPS), rtol=0, atol=1e-12), \
        "advantage = (reward - group mean) / (population std + ADV_EPS); is the std the sample std (ddof=1)?"
    assert np.all(adv[4:] == 0.0), "a group whose rewards are all equal has advantage exactly zero"
    assert adv.shape == (8,), "return one flat advantage per reward"
    given = np.array([3.0, 1.0, 2.0, 2.0])
    group_advantages(given, 2)
    zero_variance_fraction(given, 2)
    assert np.array_equal(given, [3.0, 1.0, 2.0, 2.0]), \
        "do not change the rewards you are given: np.asarray returns the caller's own array, so work on a new one, not in place"
    two = group_advantages([1.0, 0.0, 3.0, 5.0], 2)
    assert np.allclose(two, [0.5 / (0.5 + ADV_EPS), -0.5 / (0.5 + ADV_EPS), -1 / (1 + ADV_EPS), 1 / (1 + ADV_EPS)], rtol=0, atol=1e-12), \
        "the mean and std are per group, not over the whole batch"
    assert abs(group_advantages([1.0, 0.0], 2)[0] - 0.5 / (0.5 + ADV_EPS)) < 1e-12, "add ADV_EPS to the standard deviation"
    assert zero_variance_fraction([1.0, 1.0, 0.0, 1.0], 2) == 0.5, "count groups whose rewards are all equal"
    assert zero_variance_fraction([0.0, 0.1, 0.0, 0.0], 2) == 0.5, "a gap of 0.1 is a difference: compare exactly"
    assert zero_variance_fraction([0.5, 0.5 + 1e-12], 2) == 0.0, "even a gap of 1e-12 is a difference: no tolerance"
    assert zero_variance_fraction([0.0, 0.0, 1.0, 1.0], 2) == 1.0, "equal within each group counts, even if the groups differ"
    assert zero_variance_fraction([0.0, 0.0, 1.0, 0.0, 2.0, 2.0, 2.0, 2.0], 4) == 0.5, \
        "a group is tied only when every reward in it is equal: the odd one out can sit anywhere, not just second or last"
    assert type(zero_variance_fraction([1.0, 1.0], 2)) is float, \
        "return a plain Python float (wrap it in float()), not a numpy float"
    print("exercise 6 looks right")


_try("exercise 6", _check_advantages)

## 9. Exercise 7: the gradient of a tabular softmax policy

Each row of `table` holds the logits for one state. For one decision (state `s`, action `a`) the
gradient of `log π(a | s)` with respect to row `s` is `onehot(a) - softmax(table[s])`, and zero for every
other row. Write `policy_gradient(table, states, actions, weights)`: the gradient, with respect to `table`,
of `sum_i weights[i] * log π(actions[i] | states[i])`. The same state appears many times in a batch, so the
contributions to its row **add up**. Do not change `table`. `states`, `actions` and `weights` may each be a
plain Python list or a numpy array (exercise 8 passes lists). (Moving the table along this gradient makes
the actions with positive weight more likely and those with negative weight less likely.)

<details><summary>Hint 1: what to think about</summary>

Which array has the shape of `table`? Each decision contributes a whole row's worth of numbers to one
row. What goes wrong when two decisions share a state and you assign instead of add?
</details>

<details><summary>Hint 2: the approach in words</summary>

Start from zeros. For every decision compute the softmax of its row, subtract it from a one-hot of the action,
scale by the weight, and add the result into that state's row. `np.add.at` adds repeated indices correctly.
</details>

In [ ]:
def policy_gradient(table: np.ndarray, states, actions, weights) -> np.ndarray:
    """d/d table of sum_i weights[i] * log softmax(table[states[i]])[actions[i]].

    Example (a flat row of four logits, one decision with weight 2 on action 1):
        >>> g = policy_gradient(np.zeros((2, 4)), [0], [1], [2.0])
        >>> g[0].tolist(), g[1].tolist()
        ([-0.5, 1.5, -0.5, -0.5], [0.0, 0.0, 0.0, 0.0])

    Returns:
        An array of the same shape as `table`.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement policy_gradient")


def _check_policy_gradient() -> None:
    rng = np.random.default_rng(11)
    table = rng.normal(size=(5, N_ACTIONS))
    states, actions = [3, 1, 3, 3, 0], [2, 8, 2, 5, 0]
    weights = np.array([0.7, -1.2, 0.4, 2.0, -0.3])

    def objective(t):
        z = t - t.max(axis=1, keepdims=True)
        logp = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
        return float(sum(w * logp[s, a] for s, a, w in zip(states, actions, weights)))

    grad = policy_gradient(table, states, actions, weights)
    assert grad.shape == table.shape, "return an array shaped like the table"
    numeric = np.zeros_like(table)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            plus, minus = table.copy(), table.copy()
            plus[i, j] += 1e-5
            minus[i, j] -= 1e-5
            numeric[i, j] = (objective(plus) - objective(minus)) / 2e-5
    worst = np.abs(grad - numeric).max()
    assert worst < 1e-6, (f"the gradient is off from a finite-difference check by {worst:.1e}: rows are onehot(action) - "
                          "softmax(row), scaled by the weight; when a state repeats, its contributions add")
    assert np.all(grad[2] == 0) and np.all(grad[4] == 0), "states that do not appear in the batch have a zero gradient"
    as_lists = policy_gradient(table, list(states), list(actions), weights.tolist())
    assert np.allclose(as_lists, grad, rtol=0, atol=1e-12), "states, actions and weights may be plain lists: np.asarray them first"
    before = table.copy()
    policy_gradient(table, states, actions, weights)
    assert np.array_equal(table, before), "do not change the table you are given"
    big = np.zeros((1, 3))
    big[0, 1] = 1000.0
    assert np.all(np.isfinite(policy_gradient(big, [0], [0], [1.0]))), "a logit of 1000 overflows exp: shift by the row maximum"
    print("exercise 7 looks right")


_try("exercise 7", _check_policy_gradient)

## 10. Exercise 8: one group-relative update

Put the pieces together. `grpo_update(table, episodes, group_size, lr)` takes a batch of episodes ordered
group after group (as in exercise 6) and returns the **new** table:

1. the advantage of each episode from its `reward`, by `group_advantages`;
2. every step of an episode gets the weight `advantage / steps / len(episodes)`: the advantage is shared across
   the episode's steps and averaged over its length (so a long episode does not shout louder than a short one),
   then averaged over the batch;
3. `policy_gradient` over all steps of all episodes, with those weights;
4. return `table + lr * gradient`. The table you were given is not changed.

This is a single update with no importance ratio and no clipping: the batch was sampled from the table it
updates, so the ratio would be one everywhere. (T04-L01 builds the clipped version.)

<details><summary>Hint 1: what to think about</summary>

Every step of one episode carries the same advantage, so you need a flat list of states, actions and weights
with the episode's values repeated once per step. What is the weight per step, and what do you divide by?
</details>

<details><summary>Hint 2: the approach in words</summary>

Get the advantages. Loop over episodes and their advantages, extend three flat lists (states, actions, and the
weight repeated `steps` times), call `policy_gradient`, and add `lr` times it to a copy of the table.
</details>

In [ ]:
def grpo_update(table: np.ndarray, episodes: list, group_size: int, lr: float) -> np.ndarray:
    """One group-relative policy-gradient step on a batch of episodes.

    Example (two episodes in one group: a one-step success and a two-step failure; the success is
    pushed up, the failure pushed down, its weight shared over its own steps):
        >>> good = Episode([0], [1], ["solved"], 1.0, 1, True, False)
        >>> bad = Episode([0, 0], [2, 3], ["error", "gave_up"], 0.0, 2, False, False)
        >>> new = grpo_update(np.zeros((1, 4)), [good, bad], 2, 1.0)
        >>> bool(new[0, 1] > 0 and new[0, 2] < 0 and new[0, 3] < 0)
        True

    Returns:
        A new array shaped like `table`.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement grpo_update")


def _check_grpo_update() -> None:
    good = Episode([0], [1], ["solved"], 1.0, 1, True, False)
    bad = Episode([1, 2], [2, 3], ["error", "gave_up"], 0.0, 2, False, False)
    table = np.zeros((3, 4))
    new = grpo_update(table, [good, bad], 2, 1.0)
    assert new is not table and np.all(table == 0), "return a new table: the one you were given must not change"
    adv = group_advantages([1.0, 0.0], 2)
    assert abs(new[0, 1] - adv[0] / 1 / 2 * 0.75) < 1e-9, \
        "the success has advantage A over 1 step; the batch has 2 episodes: weight A / steps / len(episodes)"
    assert abs(new[2, 3] - adv[1] / 2 / 2 * 0.75) < 1e-9, "the failure's weight is shared over its 2 steps"
    assert new[1, 2] < 0 < new[0, 1], "the failure's action should be pushed down, the success's pushed up"
    twice = grpo_update(table, [good, bad, good, bad], 2, 1.0)
    assert np.allclose(twice, new, rtol=0, atol=1e-12), \
        "divide by len(episodes), the whole batch: the same batch twice over must give the same step"
    tied = grpo_update(table, [good, good], 2, 1.0)
    assert np.all(tied == 0), "a group whose rewards are tied teaches nothing: the table should not move"
    half = grpo_update(table, [good, bad], 2, 0.5)
    assert np.allclose(half - table, (new - table) / 2, rtol=0, atol=1e-12), "the step is proportional to lr"
    other = Episode([3], [0], ["solved"], 1.0, 1, True, False)
    four = grpo_update(np.zeros((4, 4)), [good, bad, other, other], 2, 1.0)
    assert np.all(four[3] == 0), ("the advantages are per group of group_size episodes: the second group is tied, so its "
                                  "state's row must not move, whatever the first group did")
    some = Episode([0, 0], [1, 3], ["repeat", "gave_up"], 0.1, 2, False, False)
    none = Episode([1, 1], [2, 3], ["error", "gave_up"], 0.0, 2, False, False)
    graded = grpo_update(np.zeros((3, 4)), [some, none], 2, 1.0)
    assert graded[0, 1] > 0 > graded[1, 2], \
        "the advantage comes from each episode's reward, not from whether it was solved (neither episode here was)"
    print("exercise 8 looks right")


_try("exercise 8", _check_grpo_update, needs=("exercise 6", "exercise 7"))

## 11. Exercise 9: the training loop

`train_grpo(mode, seed, iterations, group_size, lr)` starts from an all-zero table (a uniform policy) and
makes `iterations` updates. Each iteration it plays `group_size` episodes on **every** task, group after
group (task 0's group first, ... , task 19's), each group drawn from the generator
`iteration_rng(seed, iteration, task)` (below), one generator per group, used for its episodes in order.
It records the iteration's statistics **before** the update, from the batch it just sampled, then updates.
It returns `(table, history)`, where `history` is a list with one dict per iteration and the keys
`"reward"` (mean episode reward), `"success"` (fraction of episodes with `solved`), `"steps"` (mean
steps) and `"zero_var"` (`zero_variance_fraction` of the batch's rewards).

<details><summary>Hint 1: what to think about</summary>

Which environment object do the episodes need, and what does its reward mode do to the numbers you record?
The statistics describe the batch that the update is about to learn from.
</details>

<details><summary>Hint 2: the approach in words</summary>

Make one `ToolEnv(mode)` and a zero table. In each iteration, loop over tasks, and for each task
make its generator and collect `group_size` episodes with `run_episode`. Record the four statistics from the
flat list of episodes, then replace the table with `grpo_update(...)`.
</details>

In [ ]:
def iteration_rng(seed: int, iteration: int, task: int) -> np.random.Generator:
    """The generator for one group of one iteration of one run. Fixed by (SEED, seed, iteration, task)."""
    return np.random.default_rng([SEED, seed, iteration, task])


def train_grpo(mode: str, seed: int, iterations: int = ITERATIONS, group_size: int = GROUP, lr: float = LR):
    """Train a table policy with group-relative updates in the reward mode `mode`.

    Example:
        >>> table, history = train_grpo("outcome", 0, iterations=2, group_size=2)
        >>> table.shape, len(history), sorted(history[0])
        ((24, 9), 2, ['reward', 'steps', 'success', 'zero_var'])

    Returns:
        (table, history) as described above.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement train_grpo")


def _check_train() -> None:
    table, history = train_grpo("outcome", 0, iterations=3, group_size=2)
    assert table.shape == (N_STATES, N_ACTIONS), "return the table first, then the history"
    assert len(history) == 3 and all(set(h) == {"reward", "success", "steps", "zero_var"} for h in history), \
        "history is one dict per iteration with the keys reward, success, steps, zero_var"
    env = ToolEnv("outcome")
    first = []
    for t in range(N_TASKS):
        rng = iteration_rng(0, 0, t)
        first.extend(run_episode(env, np.zeros((N_STATES, N_ACTIONS)), t, rng) for _ in range(2))
    assert abs(history[0]["reward"] - np.mean([e.reward for e in first])) < 1e-12 and \
        abs(history[0]["steps"] - np.mean([e.steps for e in first])) < 1e-12, \
        "iteration 0 plays the all-zero table, with one iteration_rng(seed, iteration, task) per group, tasks in order"
    assert abs(history[0]["success"] - np.mean([e.solved for e in first])) < 1e-12, \
        "success is the fraction of EPISODES with solved"
    assert history[0]["zero_var"] == zero_variance_fraction([e.reward for e in first], 2), \
        "zero_var is zero_variance_fraction of the batch's rewards, in groups of the group_size you were given"
    assert np.abs(table).max() > 0, "the table should have moved away from zero"
    moved = grpo_update(np.zeros((N_STATES, N_ACTIONS)), first, 2, LR)
    second = []
    for t in range(N_TASKS):
        rng = iteration_rng(0, 1, t)
        second.extend(run_episode(env, moved, t, rng) for _ in range(2))
    for key, value in (("reward", np.mean([e.reward for e in second])), ("success", np.mean([e.solved for e in second])),
                       ("steps", np.mean([e.steps for e in second]))):
        assert abs(history[1][key] - value) < 1e-12, \
            (f"iteration 1 ({key}) plays the table after ONE update, grpo_update(zero table, iteration 0's batch, group_size, "
             "lr), with iteration_rng(seed, 1, task): carry the table forward and pass the iteration number")
    assert history[1]["zero_var"] == zero_variance_fraction([e.reward for e in second], 2), "zero_var of iteration 1's batch"
    after_two = train_grpo("outcome", 0, iterations=2, group_size=2)[0]
    assert np.allclose(after_two, grpo_update(moved, second, 2, LR), rtol=0, atol=1e-12), \
        "each update starts from the table the previous one returned (not from zeros), with group_size and lr"
    empty_table, empty = train_grpo("outcome", 0, iterations=0, group_size=2)
    assert empty == [] and empty_table.shape == (N_STATES, N_ACTIONS) and np.all(empty_table == 0), \
        "zero iterations: the all-zero table and an empty history"
    again = train_grpo("outcome", 0, iterations=3, group_size=2)
    assert np.array_equal(again[0], table) and again[1] == history, "the same arguments must give the same run"
    other = train_grpo("outcome", 1, iterations=3, group_size=2)
    assert not np.array_equal(other[0], table), "a different seed should give a different run"
    calls = train_grpo("calls", 0, iterations=1, group_size=2)[1]
    first_calls = []
    for t in range(N_TASKS):
        rng = iteration_rng(0, 0, t)
        first_calls.extend(run_episode(ToolEnv("calls"), np.zeros((N_STATES, N_ACTIONS)), t, rng) for _ in range(2))
    assert abs(calls[0]["reward"] - np.mean([e.reward for e in first_calls])) < 1e-12, \
        "train in the reward mode you were given: build the environment as ToolEnv(mode)"
    assert abs(calls[0]["success"] - np.mean([e.solved for e in first_calls])) < 1e-12, \
        ("success is the fraction of episodes with solved, not with a positive reward: in outcome mode the two agree, "
         "in calls mode they differ")
    one = train_grpo("outcome", 0, iterations=1, group_size=2)[0]
    two = train_grpo("outcome", 0, iterations=1, group_size=2, lr=2 * LR)[0]
    assert np.allclose(two, 2 * one, rtol=0, atol=1e-12), \
        "pass lr on to grpo_update: from the zero table, one iteration at twice the learning rate moves the table twice as far"
    print("exercise 9 looks right")


_try("exercise 9", _check_train, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5",
                                        "exercise 6", "exercise 7", "exercise 8"))

## 12. Exercise 10: what to measure

A training reward says what the agent was paid, not what you wanted. So the metrics come from the **events**,
not from the reward. Write `episode_metrics(episodes)` returning a dict with three plain Python floats (not numpy
floats):

- `"success_rate"`: the fraction of episodes with `solved`;
- `"mean_steps"`: the mean number of steps;
- `"hack_rate"`: the fraction of episodes that are **truncated** and contain at least `HACK_REPEATS` events equal to
  `"repeat"`. (An episode that submitted is never counted, however many repeats it made.)

<details><summary>Hint 1: what to think about</summary>

Three questions about each episode, answered from its fields: did it end in success; how long was it;
did it run into the cap while repeating calls? What is the difference between a repeat count of
exactly the threshold and one below it?
</details>

<details><summary>Hint 2: the approach in words</summary>

Take means over the episodes of: the `solved` flags, the `steps`, and a flag that is true when `truncated`
holds and the number of `"repeat"` events (use the list's `count`) reaches the threshold.
</details>

In [ ]:
def episode_metrics(episodes: list) -> dict:
    """Success rate, mean steps and hack rate of a list of episodes.

    Example:
        >>> win = Episode([0], [1], ["solved"], 1.0, 1, True, False)
        >>> loop = Episode([0] * 4, [1] * 4, ["advanced", "repeat", "repeat", "repeat"], 0.4, 4, False, True)
        >>> episode_metrics([win, loop])
        {'success_rate': 0.5, 'mean_steps': 2.5, 'hack_rate': 0.5}

    Returns:
        {"success_rate": float, "mean_steps": float, "hack_rate": float}
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement episode_metrics")


def _check_metrics() -> None:
    win = Episode([0, 0], [1, 8], ["advanced", "solved"], 1.0, 2, True, False)
    loop = Episode([0] * 5, [1] * 5, ["advanced"] + ["repeat"] * 4, 0.5, 5, False, True)
    edge = Episode([0] * 5, [1] * 5, ["advanced", "repeat", "repeat", "repeat", "error"], 0.4, 5, False, True)
    below = Episode([0] * 4, [1] * 4, ["advanced", "repeat", "repeat", "error"], 0.3, 4, False, True)
    submitted = Episode([0] * 5, [1] * 5, ["repeat"] * 4 + ["gave_up"], 0.4, 5, False, False)
    m = episode_metrics([win, loop, edge, below, submitted])
    assert m["success_rate"] == 0.2, "success_rate is the fraction of episodes with solved true"
    assert m["mean_steps"] == (2 + 5 + 5 + 4 + 5) / 5, "mean_steps is the mean of steps"
    assert m["hack_rate"] == 0.4, ("hack_rate counts truncated episodes with at least HACK_REPEATS 'repeat' events: "
                                   "exactly the threshold counts, one below does not, and a submitted episode never does")
    assert episode_metrics([win])["hack_rate"] == 0.0 and episode_metrics([loop])["hack_rate"] == 1.0, "check a single episode"
    assert all(type(v) is float for v in m.values()), \
        "return a plain Python float (wrap it in float()), not a numpy float, for each of the three values"
    calls_win = Episode([0, 0, 0], [1, 2, 8], ["advanced", "advanced", "solved"], 2 * CALL_REWARD, 3, True, False)
    paid_loop = Episode([0] * MAX_STEPS, [1] * MAX_STEPS, ["advanced"] + ["repeat"] * (MAX_STEPS - 1),
                        MAX_STEPS * CALL_REWARD, MAX_STEPS, False, True)
    assert calls_win.reward != SOLVED_REWARD and paid_loop.reward > SOLVED_REWARD
    assert episode_metrics([calls_win, calls_win, paid_loop])["success_rate"] == 2 / 3, \
        (f"success_rate is the fraction of episodes with solved, read from the events, not from the reward: in calls "
         f"mode a solve pays {2 * CALL_REWARD:.1f} and a loop pays {MAX_STEPS * CALL_REWARD:.1f}")
    print("exercise 10 looks right")


_try("exercise 10", _check_metrics)

## 13. Measurement 1: what each reward pays a perfect agent, and a looping one

Before any learning, ask the environment itself. Two scripted agents play every task in each reward
mode: the **perfect** agent (the right tool for every pending sub-goal, then submit) and a **looping** agent
(the right first tool, then the same tool again until the step cap, never submitting). Nothing is trained.
The table is the answer to "what would an agent that maximises this reward do?".

In [ ]:
def _scripted_total(mode: str, kind: str) -> float:
    env = ToolEnv(mode)
    totals = []
    for task in range(N_TASKS):
        obs = env.reset(task_id=task)
        first, total = None, 0.0
        while not obs.done:
            action = oracle_action(obs) if (kind == "perfect" or first is None) else first
            first = action if first is None else first
            obs = env.step(action)
            total += obs.reward
        totals.append(total)
    return float(np.mean(totals))


def _show_scripted_rewards() -> None:
    print(f"{'reward mode':<12} {'perfect agent earns':>20} {'looping agent earns':>20}   who is paid more")
    for mode in MODES:
        perfect, looping = _scripted_total(mode, "perfect"), _scripted_total(mode, "loop")
        who = "the looping agent" if looping > perfect else ("the perfect agent" if perfect > looping else "neither")
        print(f"{mode:<12} {perfect:>20.2f} {looping:>20.2f}   {who}")


_try("scripted rewards", _show_scripted_rewards, needs=("exercise 1", "exercise 2", "exercise 3"))

## 14. Measurement 2: the untrained policy

The all-zero table is a uniform policy: it picks each of the nine actions equally, whatever the state. The
helper below evaluates a table on every task, many episodes each, with generators fixed by the task alone,
so two tables are compared on the same random numbers. Success, average steps and the hack rate all come from
your `episode_metrics`.

In [ ]:
def evaluate_policy(table: np.ndarray) -> dict:
    """Play EVAL_PER_TASK episodes on every task and return episode_metrics of all of them."""
    env = ToolEnv("outcome")      # the mode does not matter: the metrics read events, not rewards
    episodes = []
    for task in range(N_TASKS):
        rng = np.random.default_rng([SEED, 99, task])
        episodes.extend(run_episode(env, table, task, rng) for _ in range(EVAL_PER_TASK))
    return episode_metrics(episodes)


def _span(values) -> str:
    v = np.asarray(values, dtype=float)
    return f"{v.mean():.2f} [{v.min():.2f}, {v.max():.2f}]"


def _show_baseline() -> None:
    m = evaluate_policy(np.zeros((N_STATES, N_ACTIONS)))
    shortest = np.mean([len(chain) + 1 for _, chain in TASKS])
    print(f"uniform policy over {N_TASKS * EVAL_PER_TASK} episodes: success {m['success_rate']:.3f}, "
          f"mean steps {m['mean_steps']:.2f}, hack rate {m['hack_rate']:.3f}")
    print(f"for comparison, the shortest possible episode averages {shortest:.2f} steps")


_try("untrained policy", _show_baseline, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5", "exercise 10"))

## 15. Measurement 3: training on the outcome reward

Now train: the fixed seeds in `SEEDS`, the outcome reward, `ITERATIONS` iterations of one group per task. Each line below
is the mean over the runs, with the lowest and highest run in brackets. Two columns deserve attention beyond the success rate:
the **average steps** (an episode ends when the agent submits, so the steps depend on what the policy has learned as well
as on the success rate: read how the column moves instead of expecting it to follow success) and the fraction of
**zero-variance groups**, the groups in which every episode earned the same reward, so the update gets no signal from them.

In [ ]:
_RUNS: dict = {}


def get_run(mode: str, seed: int, group: int = GROUP):
    """Train once per (mode, seed, group) and remember the result for the cells below."""
    key = (mode, seed, group)
    if key not in _RUNS:
        _RUNS[key] = train_grpo(mode, seed, ITERATIONS, group, LR)
    return _RUNS[key]


def _curves(mode: str, group: int = GROUP, seeds=SEEDS) -> dict:
    runs = [get_run(mode, s, group)[1] for s in seeds]
    return {k: np.array([[h[k] for h in run] for run in runs]) for k in ("reward", "success", "steps", "zero_var")}


def _show_learning() -> None:
    _RUNS.clear()                      # cells below reuse these runs: re-run this cell after you change your code
    c = _curves("outcome")
    marks = [i for i in (0, 10, 20, 30, 40, ITERATIONS - 1) if i < ITERATIONS]
    print(f"{'iteration':>9}  {'success':>20}  {'mean steps':>20}  {'zero-variance groups':>22}")
    for i in marks:
        print(f"{i:>9}  {_span(c['success'][:, i]):>20}  {_span(c['steps'][:, i]):>20}  {_span(c['zero_var'][:, i]):>22}")
    mean_success = c["success"].mean(axis=0)
    reached = np.nonzero(mean_success >= 0.9)[0]
    zv = c["zero_var"].mean(axis=0)
    print(f"mean success first reaches 0.9 at iteration {reached[0]}" if len(reached) else
          "mean success never reaches 0.9 in this run length")
    print(f"zero-variance groups: {zv[0]:.2f} at the start, lowest {zv.min():.2f} at iteration {int(zv.argmin())}, "
          f"{zv[-1]:.2f} at the end")
    if 0 < int(zv.argmin()) < len(zv) - 1 and zv[0] > zv.min() and zv[-1] > zv.min():
        print("the fraction of tied groups falls and then rises again: early groups are tied because nobody succeeds, "
              "late groups because everybody does")
    else:
        print("the fraction of tied groups does not fall and rise again on these runs")
    finals = [evaluate_policy(get_run("outcome", s)[0]) for s in SEEDS]
    print("trained policies, evaluated on the same episodes as the untrained one:")
    print(f"  success {_span([m['success_rate'] for m in finals])}   mean steps {_span([m['mean_steps'] for m in finals])}"
          f"   hack rate {_span([m['hack_rate'] for m in finals])}")


_try("outcome training", _show_learning,
     needs=tuple(f"exercise {k}" for k in range(1, 11)))

A group in which every episode failed and a group in which every episode succeeded look the same to the update:
both are tied, so both teach nothing. Read the zero-variance column next to the success column, never alone.

## 16. Measurement 4: the same training, a different reward

The environment decides what is paid. Now train the same code, on the same tasks and seeds, in the three
reward modes. Success and hack rate are always read from events (the outcome the author of the tasks
wanted), not from the reward being trained on. The training reward is the mean over the last five iterations.

In [ ]:
def _show_three_modes() -> None:
    print(f"{'reward mode':<12} {'train reward':>20} {'success':>20} {'mean steps':>20} {'hack rate':>20}")
    stats = {}
    for mode in MODES:
        finals = [evaluate_policy(get_run(mode, s)[0]) for s in SEEDS]
        train_reward = [np.mean([h['reward'] for h in get_run(mode, s)[1][-5:]]) for s in SEEDS]
        stats[mode] = (np.mean([m['success_rate'] for m in finals]), np.mean([m['hack_rate'] for m in finals]))
        print(f"{mode:<12} {_span(train_reward):>20} {_span([m['success_rate'] for m in finals]):>20} "
              f"{_span([m['mean_steps'] for m in finals]):>20} {_span([m['hack_rate'] for m in finals]):>20}")
    print(f"step cap {MAX_STEPS}, so the most the calls mode can pay is {MAX_STEPS * CALL_REWARD:.2f} per episode")
    outcome, calls, progress = stats["outcome"], stats["calls"], stats["progress"]
    print("calls mode:", "the agents that were paid per call solve the tasks LESS often than the outcome-trained agents "
          f"({calls[0]:.2f} against {outcome[0]:.2f}) and are caught looping in {calls[1]:.2f} of their episodes"
          if calls[0] < outcome[0] else "the agents trained on calls solve at least as many tasks as the outcome-trained agents here")
    print("progress mode:", "its success rate is within 0.05 of the outcome mode's and its hack rate is "
          f"{progress[1]:.2f}" if abs(progress[0] - outcome[0]) <= 0.05 else
          f"its success rate ({progress[0]:.2f}) differs from the outcome mode's ({outcome[0]:.2f}) by more than 0.05; "
          f"its hack rate is {progress[1]:.2f}")


_try("three rewards", _show_three_modes, needs=tuple(f"exercise {k}" for k in range(1, 11)))

## 17. Reading the hack

A rising reward curve and a falling success curve are the signature. The cell below prints the calls-mode
training curve of one run (training reward, and success read from events), then the actual tool calls
of one episode from that run's policy next to one from the outcome-trained policy, on the same task.

In [ ]:
def _trace(table: np.ndarray, task: int, label: str) -> None:
    ep = run_episode(ToolEnv("outcome"), table, task, np.random.default_rng([SEED, 7, task]))
    calls = " ".join(("submit" if a == SUBMIT else TOOLS[a]) + ("" if e in ("advanced", "solved") else f"({e})")
                     for a, e in zip(ep.actions, ep.events))
    print(f"{label}: {ep.steps} steps, solved {ep.solved}, truncated {ep.truncated}\n    {calls}")


def _show_trace() -> None:
    run = get_run("calls", SEEDS[0])
    print(f"calls mode, seed {SEEDS[0]}: training reward and true success by iteration")
    for i in (0, 5, 10, 20, ITERATIONS - 1):
        h = run[1][i]
        print(f"  iteration {i:>2}: reward {h['reward']:.2f}   success {h['success']:.2f}   steps {h['steps']:.2f}")
    _trace(run[0], 0, "trained on calls  ")
    _trace(get_run("outcome", SEEDS[0])[0], 0, "trained on outcome")


_try("hack trace", _show_trace, needs=tuple(f"exercise {k}" for k in range(1, 11)))

The loop is not a bug in the trainer or the policy. The environment pays `CALL_REWARD` for a call that worked and nothing
for finishing, so finishing is the one thing that *costs* reward: it stops the episode and the payments with it.
The `progress` mode pays for an advance, which can happen once per sub-goal, and so cannot be farmed.
The fix is a change to the environment, not to the learning algorithm.

## 18. Measurement 5: group size

The group is the baseline: with `G` episodes per task, a task is only informative if the group contains
both a success and a failure. A small group is more likely to tie, so read the zero-variance column as a mean over the run. The table runs the outcome mode
with several group sizes on the first three seeds. The iterations are the same, so a larger group also spends more episodes.

In [ ]:
def _show_group_sizes() -> None:
    seeds = SEEDS[:3]
    print(f"{'group size':>10} {'episodes/iteration':>19} {'success at iteration 20':>25} {'success at the end':>20} {'zero-variance, mean':>20}")
    rows = {}
    for g in (2, 4, GROUP, 16):
        c = _curves("outcome", g, seeds)
        rows[g] = c["success"].mean(axis=0)
        print(f"{g:>10} {g * N_TASKS:>19} {_span(c['success'][:, 20]):>25} {_span(c['success'][:, -5:].mean(axis=1)):>20} "
              f"{c['zero_var'].mean():>20.2f}")
    print("success at iteration 20 rises with the group size" if rows[2][20] < rows[4][20] < rows[GROUP][20] < rows[16][20]
          else "success at iteration 20 does not rise steadily with the group size on these seeds")


_try("group sizes", _show_group_sizes, needs=tuple(f"exercise {k}" for k in range(1, 11)))

## 19. What this toy does not show

- **The model is a table.** A language model picks tools from text, shares parameters across states, and can
  generalise or fail to. Nothing here says how that changes the optimisation.
- **The tools are a dictionary.** Real tools fail in messy ways, take arguments, return long outputs, and cost
  time and money. This environment has one deterministic dynamics and a pending sub-goal handed to the agent.
- **The loophole was planted, and it is a single loophole.** Real reward hacking is found by the agent in places
  the author did not think of. What carries over is the method: measure the outcome you want from events the
  agent cannot see, next to the reward it is trained on.
- **A few seeds are few.** The brackets show the lowest and the highest of the runs; they do not promise what the
  next run will do.
- **Interfaces move.** The conventions here are those of one release of one project, read on one date; the sources
  in `claims.yaml` say which. Re-read them before you build on them.

## 20. Common mistakes

- **Counting a later sub-goal's tool as progress.** The dynamics only advance on the tool for the *pending* sub-goal.
- **Truncation and termination in one flag.** A submit on the last step is not a truncation; if the two are
  merged, a success at the cap is reported as a cut-off.
- **Recording the state after the step.** The policy acts on the state it saw, so the gradient must use that state.
- **Averaging the wrong way.** The loss is averaged over the steps of an episode and over the batch; dividing by the
  number of steps in the whole batch changes the weights of short episodes.
- **Reading the training reward as success.** Section 17: the reward can rise toward its maximum while the thing you
  wanted falls to zero. Measure the outcome from events.
- **Comparing single runs.** Two seeds of the same code differ. Compare on the same seeds and report the spread.

In [ ]:
def _show_tied_groups() -> None:
    adv = group_advantages([0.0] * GROUP + [1.0] + [0.0] * (GROUP - 1), GROUP)
    print(f"a group of {GROUP} with no success: advantages {adv[:GROUP].tolist()}")
    print(f"a group of {GROUP} with one success: the success gets {adv[GROUP]:.2f}, the others {adv[GROUP + 1]:.2f}")


_try("tied group", _show_tied_groups, needs=("exercise 6",))

## 21. Self-check

1. Every episode in a group earns exactly the same reward. What does the group-relative update do with that group?
   - (a) It gives every episode a very large advantage, because the standard deviation is zero
   - (b) It returns `nan` because of the division by the zero standard deviation, so the group has to be removed by hand
   - (c) It gives every episode advantage zero, so the group moves the table nowhere
   - (d) It gives every episode the group's own reward as its advantage, so the whole group is pushed up together

2. An agent trained in an environment that pays for every working tool call reaches a training reward close to the maximum
   while its task success falls. What is the most accurate description?
   - (a) The trainer has a bug that flips the sign of the gradient, so it climbs the loss instead of descending it
   - (b) The environment's reward paid for something other than the outcome, and the agent did what was paid for
   - (c) The group size was too small, so the advantages came out noisy and the policy drifted away from the task
   - (d) The policy table was too small to represent the right behaviour, so it settled on the nearest thing it could store

3. Why does the environment report `truncated` separately from the end of the task?
   - (a) Truncation is faster to compute than termination, which matters when millions of episodes are collected
   - (b) A truncated episode never has a reward, so a trainer must be told to ignore it
   - (c) It makes the observation larger, which gives the policy more to condition on
   - (d) A cut-off by the step cap says nothing about whether the task would have succeeded, so a trainer may treat it differently

4. Why can the `progress` reward not be farmed by repeating a call?
   - (a) It pays only when a call advances the task, once per sub-goal
   - (b) Repeated calls raise an error that ends the episode before any further reward can be paid
   - (c) The step cap is lower in that mode, so there is no room left to repeat a call
   - (d) Its rewards are normalised to zero mean inside every episode, so extra calls cancel out

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## What you built

An environment in the hub's shape: dynamics, three reward modes, `reset`, `step`, `state` and observations that
carry `done`, `reward` and `metadata`; episode collection with a softmax policy; the group-relative advantage, the
gradient of a tabular softmax, a batch update and a training loop; and the metrics that read the outcome from events.
With them you measured how an agent learns which tool to call, how often a group teaches nothing, and what the same
training does when the environment pays for tool calls instead of outcomes.

The step from here to a real agent keeps the environment and swaps the table for a model: the same `reset`,
`step` and reward, with a language model choosing the tool. What changes is the policy and the sampler;
the lesson cannot tell you more than that.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now: each one already printed its feedback in its own cell above, so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_transition, ()),
        ("exercise 2", _check_step_reward, ()),
        ("exercise 3", _check_env, ("exercise 1", "exercise 2")),
        ("exercise 4", _check_softmax_sample, ()),
        ("exercise 5", _check_run_episode, ("exercise 1", "exercise 2", "exercise 3", "exercise 4")),
        ("exercise 6", _check_advantages, ()),
        ("exercise 7", _check_policy_gradient, ()),
        ("exercise 8", _check_grpo_update, ("exercise 6", "exercise 7")),
        ("exercise 9", _check_train, tuple(f"exercise {k}" for k in range(1, 9))),
        ("exercise 10", _check_metrics, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.